In [ ]:
# @title CSV Input
import os, io, re, base64, pickle, numpy as np, pandas as pd, matplotlib.pyplot as plt, ipywidgets as widgets
from IPython.display import display, clear_output, Javascript

try:
  from google.colab import drive
  drive.mount('/content/drive', force_remount=False)
  SD = '/content/drive/MyDrive/Fall 2026/Data Analysis Tools and Device Firmware/'
except:
  SD = os.path.join(os.getcwd(), 'rpa_fluorimeter_logs')

os.makedirs(SD, exist_ok=True)
SF = os.path.join(SD, 'trial_state.pkl')
TC = ['A0', 'A1', 'A2', 'A3']
CC = {'A0': '#1f77b4', 'A1': '#ff7f0e', 'A2': '#2ca02c', 'A3': '#d62728'}

def list_subfolders(path):
  if not os.path.isdir(path): return []
  return sorted([d for d in os.listdir(path) if os.path.isdir(os.path.join(path, d)) and not d.startswith('.')])

def list_csvs_in_folder(path):
  if not os.path.isdir(path): return []
  return sorted([f for f in os.listdir(path) if f.lower().endswith('.csv')])

def ss():
  try:
    with open(SF, 'wb') as f:
      pickle.dump({'r': R, 's': S, 'o': O, 'v': V}, f)
  except: pass

def ls():
  if os.path.exists(SF):
    try:
      with open(SF, 'rb') as f:
        d = pickle.load(f)
      return d.get('r', {}), d.get('s', {}), d.get('o', []), d.get('v', {})
    except: pass
  return {}, {}, [], {}

if 'R' not in globals():
  R, S, O, V = ls()
else:
  if 'V' not in globals(): V = {}
if 'W' not in globals(): W = {}

def cp(t):
  display(Javascript(f'navigator.clipboard.writeText(atob("{base64.b64encode(t.encode()).decode()}"))'))

def pi(t):
  t = t.strip()
  if not t: return None
  if ',' in t and any(x in t for x in ('A0', 'Voltage', 'time')):
    try:
      d = pd.read_csv(io.StringIO(t), sep=None, engine='python')
      d.columns = d.columns.str.strip()
      return d
    except: pass
  p = r'\[(\d+:\d+)\]\s+A0=([\d\.]+)mV\s+A1=([\d\.]+)mV\s+A2=([\d\.]+)mV\s+A3=([\d\.]+)mV(?:\s+T=([\d\.]+)°C)?(?:\s+duty=(\d+)%)?'
  rows = []
  for l in t.split('\n'):
    m = re.search(p, l)
    if m:
      g = m.groups()
      rows.append({'time': g[0], 'A0': float(g[1]), 'A1': float(g[2]), 'A2': float(g[3]), 'A3': float(g[4]), 'Temp': float(g[5]) if g[5] else np.nan, 'duty': float(g[6]) if g[6] else np.nan})
  return pd.DataFrame(rows) if rows else None

def gtm(df):
  tc = next((c for c in df.columns if 'time' in c.lower() or 'sec' in c.lower()), None)
  if tc is None: return pd.Series(range(len(df)), dtype=float)
  m = []
  for v in df[tc].astype(str):
    v = v.strip()
    if ':' in v:
      try: p = v.split(':'); m.append(float(p[0]) + float(p[1]) / 60)
      except: m.append(np.nan)
    else:
      try: m.append(float(v) / 60)
      except: m.append(np.nan)
  return pd.Series(m)

def nv(y):
  r = y.max() - y.min()
  return y * 0 if r == 0 or pd.isna(r) else (y - y.min()) / r

def zv(y):
  return y - y.iloc[0] if len(y) > 0 else y

rt = widgets.Textarea(value='', placeholder='Trial titles here, one per line.', layout=widgets.Layout(width='60%', height='100px'))
rb = widgets.Button(description='Apply Order', button_style='warning', icon='sort', layout=widgets.Layout(width='140px'))
rs = widgets.Output()

def rrt(): rt.value = '\n'.join(O)

def orc(b):
  with rs:
    clear_output(wait=True)
    lines = [l.strip() for l in rt.value.strip().split('\n') if l.strip()]
    ex, seen, no, err = set(O), set(), [], []
    for l in lines:
      if l not in ex: err.append(f'❌ Unknown: "{l}"')
      elif l in seen: err.append(f'❌ Duplicate: "{l}"')
      else: seen.add(l); no.append(l)
    for t in O:
      if t not in seen: no.append(t)
    if err:
      for e in err: print(e)
      print('Order not changed.'); rrt(); return
    O.clear(); O.extend(no); rc(); ss(); rrt(); print(f'✅ Reordered: {len(O)}')

rb.on_click(orc)
rp = widgets.VBox([
    widgets.HTML('<b style="color:#ffa000;font-size:14px;">📋 Reorder Cards</b>'),
    widgets.HBox([rt, widgets.VBox([rb, rs])])
])
cn = widgets.VBox([])

def rc(): cn.children = tuple(W[t]['card'] for t in O if t in W)

def rm(tid):
  R.pop(tid, None); S.pop(tid, None); V.pop(tid, None)
  if tid in O: O.remove(tid)
  W.pop(tid, None); rc(); ss(); rrt()

def mc(tid, df, com):
  ch = [c for c in TC if c in df.columns]
  tm = gtm(df)
  st = {'id': tid}
  saved_vis = V.get(tid, {})
  cb = {c: widgets.Checkbox(value=saved_vis.get(c, True), description=c, indent=False, layout=widgets.Layout(width='70px')) for c in ch}

  xb = widgets.Button(description='✕', button_style='danger', layout=widgets.Layout(width='36px'))
  cpb = widgets.Button(description='Copy CSV', icon='copy', button_style='info', layout=widgets.Layout(width='120px'))
  ub = widgets.Button(description='▲', layout=widgets.Layout(width='36px'))
  db = widgets.Button(description='▼', layout=widgets.Layout(width='36px'))
  tn = widgets.Text(value=tid, description='Trial #:', layout=widgets.Layout(width='230px'), style={'description_width': '55px'})
  ct = widgets.Text(value=com, description='Comments:', layout=widgets.Layout(width='340px'), style={'description_width': '70px'})

  so = widgets.Output()
  po = widgets.Output()

  def _save_vis():
    V[st['id']] = {c: cb[c].value for c in ch}
    ss()

  def rst():
    with so:
      clear_output(wait=True)
      stat_rows = [{'Channel': c, 'Mean': f'{df[c].dropna().mean():.3f}', 'StdDev': f'{df[c].dropna().std():.3f}', 'P2P': f'{(df[c].dropna().max()-df[c].dropna().min()):.3f}'} for c in ch]
      tc2 = next((c for c in df.columns if 'temp' in c.lower()), None)

      tbl_html = f'<span style="font-size:13px;">Samples: {len(df)}</span><br>' + pd.DataFrame(stat_rows).to_html(index=False, escape=False, classes='dataframe')

      if tc2 is not None and df[tc2].dropna().any():
        temp_out = widgets.Output()
        with temp_out:
          fig, ax = plt.subplots(figsize=(3, 1.8), dpi=90)
          tv = df[tc2].dropna()
          tx = tm[df[tc2].notna()].reset_index(drop=True)
          ax.plot(tx, tv.reset_index(drop=True), color='#e74c3c', linewidth=1.2)
          ax.set_title(f'Temp (avg {tv.mean():.1f}°C)', fontsize=9)
          ax.set_xlabel('Time (min)', fontsize=8); ax.set_ylabel('°C', fontsize=8)
          ax.tick_params(labelsize=7); ax.grid(True, linestyle='--', alpha=0.4)
          plt.tight_layout(); plt.show(); plt.close(fig)
        display(widgets.HBox([
            widgets.HTML(tbl_html),
            temp_out
        ]))
      else:
        display(widgets.HTML(tbl_html + '<br><span style="color:#999">Temp: N/A</span>'))

  def rdp(*_):
    _save_vis()
    with po:
      clear_output(wait=True)
      sel = [c for c in ch if cb[c].value]
      fig, ax = plt.subplots(1, 2, figsize=(12, 3.8), dpi=100)
      c2 = ct.value.strip()
      tt = f'{tn.value} — {c2}' if c2 else tn.value
      fig.suptitle(tt, fontsize=11, fontweight='bold')

      for c in sel:
        mk = df[c].notna()
        x, y = tm[mk].reset_index(drop=True), df[c][mk].reset_index(drop=True)
        ax[0].plot(x, nv(y), label=c, linewidth=1.1, color=CC[c])

      ax[0].set_title('Normalized (0-1)', fontsize=10)
      ax[0].set_xlabel('Time (min)', fontsize=9); ax[0].set_ylabel('Norm. signal', fontsize=9)
      ax[0].grid(True, linestyle='--', alpha=0.5)
      if sel: ax[0].legend(fontsize=8)

      for i, c in enumerate(sel):
        mk = df[c].notna()
        x, y_raw = tm[mk].reset_index(drop=True), df[c][mk].reset_index(drop=True)
        yz = zv(y_raw)
        d = yz.iloc[-1] - yz.iloc[0]
        ax[1].plot(x, yz, label=f'{c} (Δ{d:+.1f} mV)', linewidth=1.1, color=CC[c])
        mi = int(yz.idxmax())
        ox = (i - (len(sel) - 1) / 2) * 20
        kw = dict(fontsize=7, color='white', ha='center', fontweight='bold',
                   bbox=dict(boxstyle='round,pad=0.2', fc=CC[c], alpha=0.85),
                   arrowprops=dict(arrowstyle='-', color=CC[c], lw=0.7))
        ax[1].annotate(f'{y_raw.iloc[0]:.0f}', (x.iloc[0], yz.iloc[0]), xytext=(ox, 18), textcoords='offset points', **kw)
        ax[1].annotate(f'{y_raw.iloc[mi]:.0f}', (x.iloc[mi], yz.iloc[mi]), xytext=(ox, 18), textcoords='offset points', **kw)

      ax[1].set_title('Raw (zeroed to start)', fontsize=10)
      ax[1].set_xlabel('Time (min)', fontsize=9); ax[1].set_ylabel('ΔmV', fontsize=9)
      ax[1].grid(True, linestyle='--', alpha=0.5); ax[1].margins(x=0.1, y=0.25)
      if sel: ax[1].legend(fontsize=8)

      plt.tight_layout(rect=[0, 0, 1, 0.92]); plt.show(); plt.close(fig)

  for v in cb.values(): v.observe(rdp, names='value')

  xb.on_click(lambda b: rm(st['id']))
  cpb.on_click(lambda b: cp(df.to_csv(index=False)))
  ub.on_click(lambda b: _mu(st['id']))
  db.on_click(lambda b: _md(st['id']))

  def _mu(tid):
    i = O.index(tid)
    if i > 0: O[i], O[i-1] = O[i-1], O[i]; rc(); ss(); rrt()

  def _md(tid):
    i = O.index(tid)
    if i < len(O) - 1: O[i], O[i+1] = O[i+1], O[i]; rc(); ss(); rrt()

  def _rn(w):
    old, new = st['id'], tn.value.strip()
    if not new or new == old or new in R:
      tn.value = old; return
    R[new] = R.pop(old); S[new] = S.pop(old, ''); V[new] = V.pop(old, {})
    O[O.index(old)] = new; W[new] = W.pop(old); st['id'] = new
    ss(); rrt(); rdp()

  def _rc(w):
    S[st['id']] = ct.value
    ss(); rdp()

  tn.on_submit(_rn)
  ct.on_submit(_rc)

  card = widgets.VBox([
      widgets.HTML('<hr style="margin:6px 0;">'),
      widgets.HBox([tn, ct, cpb, ub, db, xb]),
      so,
      widgets.HBox(list(cb.values())),
      po
  ])
  rst(); rdp()
  return card, {'card': card, 'state': st}

ti = widgets.Text(value=f'Trial {len(R)+1}', description='Trial #:', layout=widgets.Layout(width='45%'))
ci = widgets.Textarea(value='', description='Comments:', layout=widgets.Layout(width='90%', height='50px'))
pb = widgets.Textarea(value='', placeholder='Paste raw serial log or CSV data here...', description='Raw Data:', layout=widgets.Layout(width='90%', height='160px'))
ab = widgets.Button(description='Add (paste)', button_style='success', icon='plus', layout=widgets.Layout(width='30%'))
ts = widgets.Output()

DRIVE_ROOT = '/content/drive/MyDrive'
current_folder = [DRIVE_ROOT]

folder_dd = widgets.Dropdown(options=[], description='Subfolder:', layout=widgets.Layout(width='60%'), style={'description_width': '80px'})
folder_path_label = widgets.HTML(value=f'<b>📂 {DRIVE_ROOT}</b>')
folder_up_btn = widgets.Button(description='⬆ Up', layout=widgets.Layout(width='60px'), button_style='warning')
folder_open_btn = widgets.Button(description='📂 Open', layout=widgets.Layout(width='80px'), button_style='info')

file_select = widgets.SelectMultiple(options=[], description='CSV Files:',
    layout=widgets.Layout(width='60%', height='160px'), style={'description_width': '80px'})
select_all_btn = widgets.Button(description='Select All', icon='check-square', layout=widgets.Layout(width='110px'))
clear_sel_btn = widgets.Button(description='Clear', icon='square-o', layout=widgets.Layout(width='80px'))
refresh_btn = widgets.Button(description='🔄 Refresh', icon='refresh', layout=widgets.Layout(width='110px'))
load_drive_btn = widgets.Button(description='Add Selected from Drive', button_style='primary', icon='cloud-download', layout=widgets.Layout(width='30%'))
drive_status = widgets.Output()

def update_folder_view():
  path = current_folder[0]
  folder_path_label.value = f'<b>📂 {path.replace("/content/drive/MyDrive/", "MyDrive/")}</b>'
  subs = list_subfolders(path)
  folder_dd.options = [('— pick subfolder —', '')] + [(f'📁 {s}', s) for s in subs]
  folder_dd.value = ''
  csvs = list_csvs_in_folder(path)
  file_select.options = [(f'📄 {f}', os.path.join(path, f)) for f in csvs]
  file_select.value = []

def on_folder_open(b):
  sel = folder_dd.value
  if sel:
    current_folder[0] = os.path.join(current_folder[0], sel)
    update_folder_view()

def on_folder_up(b):
  if current_folder[0] != DRIVE_ROOT:
    current_folder[0] = os.path.dirname(current_folder[0])
    update_folder_view()

def on_refresh(b):
  update_folder_view()
  with drive_status:
    clear_output(wait=True)
    print(f'🔄 Found {len(file_select.options)} CSV(s) in current folder.')

def on_select_all(b):
  file_select.value = [v for _, v in file_select.options]

def on_clear_sel(b):
  file_select.value = []

folder_open_btn.on_click(on_folder_open)
folder_up_btn.on_click(on_folder_up)
refresh_btn.on_click(on_refresh)
select_all_btn.on_click(on_select_all)
clear_sel_btn.on_click(on_clear_sel)
update_folder_view()

def _unique_tid(base):
  if base not in R: return base
  counter = 2
  while f'{base} ({counter})' in R: counter += 1
  return f'{base} ({counter})'

def _add_trial(tid, df, comment):
  R[tid] = df; S[tid] = comment; O.insert(0, tid)
  V[tid] = {c: True for c in TC if c in df.columns}
  card, refs = mc(tid, df, S[tid])
  W[tid] = refs

def _load_single_csv(fpath):
  if not os.path.isfile(fpath): return None, f'File not found: {fpath}'
  base = os.path.splitext(os.path.basename(fpath))[0]
  tid = _unique_tid(base)
  try:
    df = pd.read_csv(fpath, sep=None, engine='python')
    df.columns = df.columns.str.strip()
  except Exception as e:
    return None, f'Failed to read {base}: {e}'
  if df.empty: return None, f'{base} is empty'
  if not any(c in df.columns for c in TC):
    try:
      with open(fpath, 'r') as f: raw = f.read()
      df2 = pi(raw)
      if df2 is not None and not df2.empty and any(c in df2.columns for c in TC):
        df = df2
      else: return None, f'{base}: no channels ({", ".join(TC)}) found'
    except: return None, f'{base}: no channels ({", ".join(TC)}) found'
  return tid, df

def oac(b):
  with ts:
    clear_output(wait=True)
    tid = ti.value.strip() or 'Unnamed Trial'
    if tid in R: print(f'❌ "{tid}" already exists.'); return
    df = pi(pb.value)
    if df is None or df.empty: print('❌ Could not parse data.'); return
    if not any(c in df.columns for c in TC): print('❌ No channels found.'); return
    _add_trial(tid, df, ci.value.strip())
    rc(); ss(); rrt()
    print(f'✅ Added {tid}'); ti.value = f'Trial {len(R)+1}'; ci.value = ''; pb.value = ''

def on_load_drive(b):
  with drive_status:
    clear_output(wait=True)
    selected = list(file_select.value)
    if not selected:
      print('❌ No files selected.'); return
    comment = ci.value.strip()
    loaded, errors = [], []
    for fpath in selected:
      result = _load_single_csv(fpath)
      if result[0] is None: errors.append(result[1])
      else:
        tid, df = result
        _add_trial(tid, df, comment)
        loaded.append((tid, len(df), os.path.basename(fpath)))
    rc(); ss(); rrt()
    if loaded:
      print(f'✅ Loaded {len(loaded)} file(s):')
      for tid, nrows, fname in loaded:
        print(f'   📄 {tid}  ({nrows} rows)  ← {fname}')
    if errors:
      print(f'\n⚠️ {len(errors)} failed:')
      for e in errors: print(f'   ❌ {e}')
    ti.value = f'Trial {len(R)+1}'; ci.value = ''
    file_select.value = []

ab.on_click(oac)
load_drive_btn.on_click(on_load_drive)

for t in O:
  if t in R:
    _, refs = mc(t, R[t], S.get(t, ''))
    W[t] = refs

rc(); clear_output()
display(widgets.HTML('<style>.dataframe th,.dataframe td{font-size:13px!important;}</style>'))
display(rp)
display(widgets.VBox([
    widgets.HTML('<b style="color:#1976d2;font-size:14px;">📁 Load CSV(s) from Google Drive</b>'),
    widgets.HTML('<span style="color:#666;font-size:12px;">Hold <b>Ctrl</b>/<b>Cmd</b> to select multiple</span>'),
    folder_path_label,
    widgets.HBox([folder_dd, folder_open_btn, folder_up_btn]),
    widgets.HBox([file_select, widgets.VBox([select_all_btn, clear_sel_btn, refresh_btn])]),
    load_drive_btn, drive_status
], layout=widgets.Layout(border='2px solid #1976d2', padding='10px', margin='8px 0', border_radius='8px')))
display(widgets.VBox([
    widgets.HTML('<b style="color:#388e3c;font-size:14px;">📋 Or Paste Raw Data</b>'),
    ti, ci, pb, ab, ts
], layout=widgets.Layout(border='2px solid #388e3c', padding='10px', margin='8px 0', border_radius='8px')))
display(cn)
rrt()
if O: print(f'📂 Loaded {len(O)} saved trial(s).')

HTML(value='<style>.dataframe th,.dataframe td{font-size:13px!important;}</style>')

📂 Loaded 16 saved trial(s).


Use the CSV Input block to paste the raw CSV data downloaded from the Pico. The Google Drive sync needs to be setup to your Google Drive directory and after granting Google Drive access, it will be possible to select CSV files directly from Drive rather than manually copy/pasting into this notebook.

In [ ]:
# @title Temperature
import numpy as np, pandas as pd, matplotlib.pyplot as plt, io, base64, ipywidgets as widgets
from IPython.display import display, clear_output, HTML as DHTML

DPI = 300
AVG_LW = 1.6

def _tv_save_fig(fig, filename='plot.png'):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=DPI, bbox_inches='tight')
    buf.seek(0)
    b64 = base64.b64encode(buf.read()).decode()
    display(DHTML(f'<a download="{filename}" href="data:image/png;base64,{b64}" '
                  f'style="color:#4fc3f7;font-size:12px">⬇ Download {filename}</a>'))

def _tv_time_minutes(df):
    tc = next((c for c in df.columns if 'time' in c.lower() or 'sec' in c.lower()), None)
    if tc is None:
        return pd.Series(np.arange(len(df)), dtype=float)
    vals = []
    for v in df[tc].astype(str):
        v = v.strip()
        if ':' in v:
            try: p = v.split(':'); vals.append(float(p[0]) + float(p[1]) / 60.0)
            except: vals.append(np.nan)
        else:
            try: vals.append(float(v) / 60.0)
            except: vals.append(np.nan)
    return pd.Series(vals, dtype=float)

def _tv_find_temp_col(df):
    for c in df.columns:
        if 'temp' in c.lower() or c.strip() == 'T':
            return c
    return None

tv_trial_select = widgets.SelectMultiple(
    options=[], description='Trials:',
    layout=widgets.Layout(width='50%', height='180px'),
    style={'description_width': '60px'})
tv_refresh_btn = widgets.Button(description='🔄 Refresh trial list', icon='refresh',
                                 layout=widgets.Layout(width='170px'))
tv_sel_all = widgets.Button(description='Select All', icon='check-square',
                             layout=widgets.Layout(width='110px'))
tv_clear_sel = widgets.Button(description='Clear', icon='square-o',
                               layout=widgets.Layout(width='80px'))

tv_show_individual = widgets.Checkbox(value=True, description='Show individual traces', indent=False)
tv_show_avg = widgets.Checkbox(value=True, description='Show averaged trace', indent=False)
tv_ss_cutoff = widgets.FloatText(value=2.5, description='Steady-state after (min):',
    style={'description_width': '150px'}, layout=widgets.Layout(width='250px'))

tv_plot_btn = widgets.Button(description='Plot Temperature Traces',
    button_style='success', icon='line-chart',
    layout=widgets.Layout(width='250px', height='38px'))
tv_out = widgets.Output()
tv_stats_label = widgets.HTML(value='')

def _tv_refresh_list(*_):
    available = []
    for tid in O:
        if tid in R:
            df = R[tid]
            tc = _tv_find_temp_col(df)
            if tc is not None and pd.to_numeric(df[tc], errors='coerce').dropna().shape[0] > 0:
                available.append(tid)
    tv_trial_select.options = [(t, t) for t in available]
    tv_trial_select.value = []

def _tv_sel_all_cb(b):
    tv_trial_select.value = [v for _, v in tv_trial_select.options]

def _tv_clear_cb(b):
    tv_trial_select.value = []

tv_refresh_btn.on_click(_tv_refresh_list)
tv_sel_all.on_click(_tv_sel_all_cb)
tv_clear_sel.on_click(_tv_clear_cb)

def _tv_plot(b):
    tv_stats_label.value = ''
    with tv_out:
        clear_output(wait=True)
        selected = list(tv_trial_select.value)
        if not selected:
            print('❌ Select at least one trial.'); return

        ss_cut = tv_ss_cutoff.value
        cmap = plt.cm.get_cmap('tab10', max(len(selected), 10))
        traces = []

        for idx, tid in enumerate(selected):
            df = R[tid]
            tc = _tv_find_temp_col(df)
            temp = pd.to_numeric(df[tc], errors='coerce')
            t_min = _tv_time_minutes(df)
            mask = temp.notna() & t_min.notna()
            if mask.sum() == 0: continue
            traces.append((t_min[mask].reset_index(drop=True),
                           temp[mask].reset_index(drop=True), tid))

        if not traces:
            print('❌ No usable temperature data.'); return

        fig, ax = plt.subplots(figsize=(10, 5), dpi=DPI)

        ax.axvline(x=ss_cut, color='grey', linestyle=':', linewidth=0.6, alpha=0.7)

        if tv_show_individual.value:
            for idx, (t, temp, label) in enumerate(traces):
                ax.plot(t, temp, linewidth=0.5, alpha=0.55, color=cmap(idx % 10))

        if tv_show_avg.value and len(traces) >= 1:
            t_common = traces[0][0].values
            stacked = np.array([temp.values for _, temp, _ in traces])
            avg = stacked.mean(axis=0)
            ax.plot(t_common, avg, 'k-', linewidth=AVG_LW, zorder=10)

            avg_mean = np.mean(avg)
            avg_overall_sd = np.std(avg)
            avg_pk2pk = np.max(avg) - np.min(avg)

            ss_mask = t_common >= ss_cut
            ss_avg = avg[ss_mask] if ss_mask.any() else avg
            ss_avg_mean = np.mean(ss_avg)
            ss_avg_sd = np.std(ss_avg)
            ss_avg_pk2pk = np.max(ss_avg) - np.min(ss_avg)

            ss_individual_pk2pk = []
            for i, (t_raw, temp_raw, tid) in enumerate(traces):
                ss_m = t_raw >= ss_cut
                if ss_m.any():
                    pk = temp_raw[ss_m].max() - temp_raw[ss_m].min()
                else:
                    pk = temp_raw.max() - temp_raw.min()
                ss_individual_pk2pk.append((pk, tid))

            ss_individual_pk2pk.sort(key=lambda x: x[0], reverse=True)
            worst_pk, worst_tid = ss_individual_pk2pk[0]

            if len(traces) > 1:
                mean_cross_sd = np.mean(stacked.std(axis=0))
                ss_stacked = stacked[:, ss_mask] if ss_mask.any() else stacked
                ss_mean_cross_sd = np.mean(ss_stacked.std(axis=0))

                stats_html = (
                    f'<div style="margin-top:6px; padding:8px 12px; '
                    f'background:#fafafa; border:1px solid #ddd; border-radius:6px; '
                    f'font-size:13px; color:#333;">'
                    f'<b>Average Trace Stats</b> &nbsp;(n = {len(traces)} trials)<br>'
                    f'&nbsp;&nbsp;• Mean temperature: <b>{avg_mean:.2f} °C</b><br>'
                    f'&nbsp;&nbsp;• SD of average trace over time: <b>{avg_overall_sd:.3f} °C</b><br>'
                    f'&nbsp;&nbsp;• Peak-to-peak of average trace: <b>{avg_pk2pk:.3f} °C</b><br>'
                    f'&nbsp;&nbsp;• Mean cross-trial SD: <b>{mean_cross_sd:.3f} °C</b><br>'
                    f'<br>'
                    f'<b>Steady-State Stats</b> &nbsp;(after {ss_cut} min)<br>'
                    f'&nbsp;&nbsp;• Mean temperature: <b>{ss_avg_mean:.2f} °C</b><br>'
                    f'&nbsp;&nbsp;• SD of average trace: <b>{ss_avg_sd:.3f} °C</b><br>'
                    f'&nbsp;&nbsp;• Pk-to-pk of average trace: <b>{ss_avg_pk2pk:.3f} °C</b><br>'
                    f'&nbsp;&nbsp;• Mean cross-trial SD: <b>{ss_mean_cross_sd:.3f} °C</b><br>'
                    f'&nbsp;&nbsp;• <span style="color:#c62828;">Largest individual trial pk-to-pk: '
                    f'<b>{worst_pk:.3f} °C</b> ({worst_tid})</span>'
                    f'</div>')
            else:
                stats_html = (
                    f'<div style="margin-top:6px; padding:8px 12px; '
                    f'background:#fafafa; border:1px solid #ddd; border-radius:6px; '
                    f'font-size:13px; color:#333;">'
                    f'<b>Average Trace Stats</b> &nbsp;(n = 1 trial)<br>'
                    f'&nbsp;&nbsp;• Mean temperature: <b>{avg_mean:.2f} °C</b><br>'
                    f'&nbsp;&nbsp;• SD of trace over time: <b>{avg_overall_sd:.3f} °C</b><br>'
                    f'&nbsp;&nbsp;• Peak-to-peak: <b>{avg_pk2pk:.3f} °C</b><br>'
                    f'<br>'
                    f'<b>Steady-State Stats</b> &nbsp;(after {ss_cut} min)<br>'
                    f'&nbsp;&nbsp;• Mean temperature: <b>{ss_avg_mean:.2f} °C</b><br>'
                    f'&nbsp;&nbsp;• SD of trace: <b>{ss_avg_sd:.3f} °C</b><br>'
                    f'&nbsp;&nbsp;• <span style="color:#c62828;">Peak-to-peak: '
                    f'<b>{ss_avg_pk2pk:.3f} °C</b></span>'
                    f'</div>')
            tv_stats_label.value = stats_html
        else:
            tv_stats_label.value = ''

        ax.set_xlabel('Time (min)', fontsize=11)
        ax.set_ylabel('Temperature (°C)', fontsize=11)
        ax.set_title('Average Temperature Over Time', fontsize=13, fontweight='bold')
        ax.grid(True, linestyle='--', alpha=0.4)

        plt.tight_layout()
        _tv_save_fig(fig, 'temperature.png')
        plt.show()
        plt.close(fig)

tv_plot_btn.on_click(_tv_plot)
_tv_refresh_list()

display(widgets.VBox([
    widgets.HTML(
        '<h3 style="color:#c62828; margin-bottom:4px;">Temperature Trace Viewer</h3>'
        '<span style="color:#666;font-size:12px;">Hold <b>Ctrl/Cmd</b> to multi-select.</span>'),
    widgets.HBox([tv_trial_select, widgets.VBox([tv_sel_all, tv_clear_sel, tv_refresh_btn])]),
    widgets.HBox([tv_show_individual, tv_show_avg]),
    tv_ss_cutoff,
    tv_plot_btn,
    tv_out,
    tv_stats_label,
], layout=widgets.Layout(border='2px solid #c62828', padding='10px', margin='12px 0', border_radius='8px')))

This provides a larger graph of the temperature recordings.

In [ ]:
# @title Overlay
import numpy as np, pandas as pd, matplotlib.pyplot as plt, io, base64, ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from collections import OrderedDict
from scipy.signal import savgol_filter

CHANNELS = ['A0','A1','A2','A3']
DPI = 300

PALETTE_8 = OrderedDict([
    ('Blue',     '#1f77b4'),
    ('Orange',   '#ff7f0e'),
    ('Green',    '#2ca02c'),
    ('Red',      '#d62728'),
    ('Purple',   '#9467bd'),
    ('Brown',    '#8c564b'),
    ('Pink',     '#e377c2'),
    ('Gray',     '#7f7f7f'),
])
PALETTE_NAMES = list(PALETTE_8.keys())
PALETTE_HEX   = list(PALETTE_8.values())

COLORS = PALETTE_HEX

LEGEND_MAP = {'0': 'NTC', '1': '10^-2', '2': '10^-3', '3': '10^-4',
              '4': '10^-5', '5': '10^-6', '6': '10^-7'}

def _ov_save_fig(fig, filename='plot.png'):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=DPI, bbox_inches='tight')
    buf.seek(0)
    b64 = base64.b64encode(buf.read()).decode()
    display(HTML(f'<a download="{filename}" href="data:image/png;base64,{b64}" '
                 f'style="color:#4fc3f7;font-size:12px">⬇ Download {filename}</a>'))

def apply_sg(y, window, polyorder):
    n = len(y)
    if n < 5: return y
    w = min(window, n)
    if w % 2 == 0: w -= 1
    if w < polyorder + 2:
        w = polyorder + 2
        if w % 2 == 0: w += 1
    if w > n: return y
    try: return pd.Series(savgol_filter(y.values, window_length=w, polyorder=polyorder), index=y.index)
    except: return y

def get_sg_params_overlay(tid):
    if ov_sg_global_cb.value:
        return True, ov_sg_global_win.value, ov_sg_global_poly.value
    return False, 25, 4

def _ov_auto_legend(group_name):
    g = group_name.strip()
    if 'water' in g.lower(): return 'Water'
    return LEGEND_MAP.get(g, '')

def _ov_get_time(df):
    tc = next((c for c in df.columns if 'time' in c.lower() or 'sec' in c.lower()), None)
    if tc is None: return pd.Series(range(len(df)), dtype=float)
    out = []
    for v in df[tc].astype(str):
        v = v.strip()
        if ':' in v:
            try: p = v.split(':'); out.append(float(p[0]) + float(p[1]) / 60)
            except: out.append(np.nan)
        else:
            try: out.append(float(v) / 60)
            except: out.append(np.nan)
    return pd.Series(out)

def _ov_norm(y):
    r = y.max() - y.min()
    return y * 0 if r == 0 or pd.isna(r) else (y - y.min()) / r

def _ov_xform(y, mode):
    if mode == 'Normalized (0-1)': return _ov_norm(y), 'Norm.'
    return y - y.min(), 'ΔmV'

def _ov_clip(x, y, tmax):
    if tmax is None: return x, y
    m = x <= tmax
    return x[m].reset_index(drop=True), y[m].reset_index(drop=True)

def _ov_trim_at_min(x, y):
    if len(y) == 0: return x, y
    i = int(np.argmin(y.values))
    return x[i:].reset_index(drop=True), y[i:].reset_index(drop=True)

def _ov_interp_avg(xs, ys):
    x0 = max(a[0] for a in xs)
    x1 = min(a[-1] for a in xs)
    if x1 <= x0: return None, None
    xc = np.linspace(x0, x1, min(len(a) for a in ys))
    return xc, np.mean([np.interp(xc, xi, yi) for xi, yi in zip(xs, ys)], axis=0)

def _ov_first_crossing(x, y, level):
    for i in range(len(y) - 1):
        if (y[i] - level) * (y[i+1] - level) <= 0:
            dy = y[i+1] - y[i]
            if dy == 0: return float(x[i])
            return float(x[i] + (level - y[i]) / dy * (x[i+1] - x[i]))
    return None

def _ov_tname(t):
    c = S.get(t, '')
    return f'{t} — {c}' if c else t

def _ov_auto_group(tid):
    s = tid.strip()
    if 'water' in s.lower(): return 'water'
    if s and s[0].isdigit():
        prefix = ''
        for ch in s:
            if ch.isdigit(): prefix += ch
            else: break
        return prefix
    return ''

def _ov_maybe_smooth(y, tid):
    enabled, win, poly = get_sg_params_overlay(tid)
    if enabled: return apply_sg(y, win, poly)
    return y

# ── single colour per trial ──
def _ov_trial_color(tid):
    """Return the hex colour chosen for a trial."""
    c = ov_configs.get(tid)
    if c and 'color' in c:
        name = c['color'].value
        return PALETTE_8.get(name, PALETTE_HEX[0])
    return PALETTE_HEX[0]

def _ov_get_channel_data(tid, mode, do_trim, tmax):
    c = ov_configs.get(tid)
    df = R.get(tid)
    if not c or df is None: return []
    sel = [k for k, v in c['ch'].items() if v.value]
    out = []
    for ch in sel:
        if ch not in df.columns: continue
        m = df[ch].notna()
        x = _ov_get_time(df)[m].reset_index(drop=True)
        y_raw = df[ch][m].reset_index(drop=True)
        y_raw = _ov_maybe_smooth(y_raw, tid)
        yt, _ = _ov_xform(y_raw, mode)
        if do_trim: x, yt = _ov_trim_at_min(x, yt)
        x, yt = _ov_clip(x, yt, tmax)
        if len(x) > 0: out.append((x, yt, ch))
    return out

def ov_build_curve(tid, mode, do_trim, tmax):
    ch_data = _ov_get_channel_data(tid, mode, do_trim, tmax)
    if not ch_data: return None
    c = ov_configs.get(tid)
    sel = [k for k, v in c['ch'].items() if v.value]
    traces = [yt for _, yt, _ in ch_data]
    x = ch_data[0][0]
    avg = pd.concat(traces, axis=1).mean(axis=1)
    if len(x) == 0: return None
    return x, avg, f'{tid}/AVG({",".join(sel)})', sel

def ov_build_channel_curves(tid, mode, do_trim, tmax):
    ch_data = _ov_get_channel_data(tid, mode, do_trim, tmax)
    return [(x.values, yt.values) for x, yt, _ in ch_data]

def ov_build_traces(tid, mode, do_trim, tmax):
    ch_data = _ov_get_channel_data(tid, mode, do_trim, tmax)
    return [(x, yt, f'{tid}/{ch}') for x, yt, ch in ch_data]

ov_configs = {}
ov_group_dropdowns = {}
ov_selected_order = []
ov_legend_edits = {}

ov_refresh_btn = widgets.Button(description='🔄 Refresh Trial List', button_style='warning',
                                 layout=widgets.Layout(width='200px', height='36px'))
ov_trial_sel = widgets.SelectMultiple(
    options=[(_ov_tname(t), t) for t in O],
    description='Trials:', layout=widgets.Layout(width='95%', height='180px'),
    style={'description_width': '70px'})
ov_select_btn = widgets.Button(description='Select Trials → Configure', button_style='info',
                                icon='arrow-right', layout=widgets.Layout(width='300px', height='38px'))
ov_cfg_lbl = widgets.HTML('')
ov_cfg_box = widgets.VBox([])

ov_mode_sel = widgets.RadioButtons(
    options=['Normalized (0-1)', 'Zeroed (ΔmV)'],
    value='Normalized (0-1)', description='Mode:',
    layout=widgets.Layout(width='220px'), style={'description_width': '55px'})
ov_trace_sel = widgets.ToggleButtons(
    options=['Avg', 'Traces'], value='Avg', description='Show:',
    layout=widgets.Layout(width='240px'),
    style={'description_width': '50px', 'button_width': '80px'})
ov_trim_cb = widgets.Checkbox(value=False, description='Trim before minimum (visual only)',
                               indent=False, layout=widgets.Layout(width='320px'))

ov_sg_global_cb = widgets.Checkbox(value=True, description='SG Smooth',
                                    indent=False, layout=widgets.Layout(width='140px'))
ov_sg_global_win = widgets.IntSlider(value=25, min=5, max=101, step=2, description='Window:',
                                      style={'description_width': '60px'},
                                      layout=widgets.Layout(width='240px'),
                                      continuous_update=False)
ov_sg_global_poly = widgets.IntSlider(value=4, min=1, max=5, step=1, description='Poly:',
                                       style={'description_width': '40px'},
                                       layout=widgets.Layout(width='200px'),
                                       continuous_update=False)

def _ov_toggle_sg(*_):
    ov_sg_global_win.disabled = not ov_sg_global_cb.value
    ov_sg_global_poly.disabled = not ov_sg_global_cb.value

ov_sg_global_cb.observe(_ov_toggle_sg, 'value')
_ov_toggle_sg()

ov_title_w = widgets.Text(value='', placeholder='auto-generated if blank',
                           description='Title:', style={'description_width': '60px'},
                           layout=widgets.Layout(width='500px'))
ov_xlabel_w = widgets.Text(value='', placeholder='Time (min)',
                            description='X label:', style={'description_width': '60px'},
                            layout=widgets.Layout(width='340px'))
ov_ylabel_w = widgets.Text(value='', placeholder='auto (Norm. / ΔmV)',
                            description='Y label:', style={'description_width': '60px'},
                            layout=widgets.Layout(width='340px'))

ov_plot_btn = widgets.Button(description='Generate Plot', button_style='primary',
                              icon='line-chart',
                              layout=widgets.Layout(width='220px', height='40px'))
ov_out_w = widgets.Output()
ov_status = widgets.Output()

def ov_refresh(b=None):
    ov_trial_sel.options = [(_ov_tname(t), t) for t in O]
    with ov_status:
        clear_output(wait=True)
        print(f'✅ {len(O)} trial(s) available')

ov_refresh_btn.on_click(ov_refresh)

def ov_rebuild_cfg():
    cards = []
    n = len(ov_selected_order)
    for i, tid in enumerate(ov_selected_order):
        df = R.get(tid)
        if df is None: continue
        chs = [c for c in CHANNELS if c in df.columns]
        saved = V.get(tid, {})

        if tid in ov_configs:
            cbs = ov_configs[tid]['ch']
            color_dd = ov_configs[tid].get('color')
        else:
            cbs = {c: widgets.Checkbox(value=saved.get(c, True), description=c, indent=False,
                                        layout=widgets.Layout(width='65px')) for c in chs}
            color_dd = None

        # one colour dropdown per trial — auto-assign based on order
        if color_dd is None:
            default_name = PALETTE_NAMES[i % len(PALETTE_NAMES)]
            color_dd = widgets.Dropdown(options=PALETTE_NAMES, value=default_name,
                                         layout=widgets.Layout(width='95px'))

        if tid not in ov_group_dropdowns:
            ov_group_dropdowns[tid] = widgets.Text(value=_ov_auto_group(tid), placeholder='group',
                                                    layout=widgets.Layout(width='120px'))
        if tid not in ov_legend_edits:
            group_val = ov_group_dropdowns[tid].value.strip() or _ov_auto_group(tid)
            auto_leg = _ov_auto_legend(group_val)
            ov_legend_edits[tid] = widgets.Text(value=auto_leg, placeholder='auto',
                                                 layout=widgets.Layout(width='180px'))

        ov_configs[tid] = {'ch': cbs, 'color': color_dd, 'ci': i}

        col = _ov_trial_color(tid)

        ub = widgets.Button(description='▲', layout=widgets.Layout(width='32px', height='26px'), disabled=i == 0)
        db = widgets.Button(description='▼', layout=widgets.Layout(width='32px', height='26px'), disabled=i == n - 1)

        def mk(t, d):
            def h(b):
                x = ov_selected_order.index(t)
                y = x + d
                if 0 <= y < len(ov_selected_order):
                    ov_selected_order[x], ov_selected_order[y] = ov_selected_order[y], ov_selected_order[x]
                ov_rebuild_cfg()
            return h

        ub.on_click(mk(tid, -1))
        db.on_click(mk(tid, 1))

        lbl = widgets.HTML(
            f'<span style="color:{col};font-weight:bold;font-size:15px">●</span> '
            f'<span style="color:#e0e0e0;font-size:12px;font-weight:600">{_ov_tname(tid)}</span>')

        r1 = widgets.HBox(
            [ub, db, lbl,
             widgets.HTML('<span style="color:#aaa;font-size:11px;margin:0 4px 0 12px">Colour:</span>'),
             color_dd] + list(cbs.values()),
            layout=widgets.Layout(align_items='center'))

        r2 = widgets.HBox([
            widgets.HTML('<span style="color:#aaa;font-size:11px;margin-right:4px">Group:</span>'),
            ov_group_dropdowns[tid],
            widgets.HTML('<span style="color:#aaa;font-size:11px;margin:0 4px 0 12px">Legend:</span>'),
            ov_legend_edits[tid],
        ], layout=widgets.Layout(margin='0 0 0 72px'))

        cards.append(widgets.VBox([r1, r2],
                                   layout=widgets.Layout(padding='6px 10px', margin='3px 0',
                                                         border=f'1px solid {col}44',
                                                         border_radius='5px')))
    ov_cfg_lbl.value = (
        f'<p style="color:#4fc3f7;font-size:13px"><b>Step 2:</b> Configure ({n} trials)</p>'
        f'<p style="color:#aaa;font-size:11px;margin:2px 0 6px">'
        f'<b>Colour</b>: applies to all channels. '
        f'<b>Group</b>: same name → combined avg. <b>Legend</b>: override label.</p>'
    ) if ov_selected_order else ''
    ov_cfg_box.children = cards

def ov_on_select(b):
    s = list(ov_trial_sel.value)
    if not s:
        ov_cfg_lbl.value = '<p style="color:#ff6b6b">❌ Select trials</p>'
        ov_cfg_box.children = []; return
    prev = [t for t in ov_selected_order if t in s]
    new = [t for t in s if t not in prev]
    ov_selected_order.clear()
    ov_selected_order.extend(prev + new)
    for t in [k for k in list(ov_configs.keys()) if k not in ov_selected_order]: del ov_configs[t]
    for t in [k for k in list(ov_group_dropdowns.keys()) if k not in ov_selected_order]: del ov_group_dropdowns[t]
    for t in [k for k in list(ov_legend_edits.keys()) if k not in ov_selected_order]: del ov_legend_edits[t]
    ov_rebuild_cfg()

ov_select_btn.on_click(ov_on_select)

def ov_get_legend(tid, auto_label):
    le = ov_legend_edits.get(tid)
    if le and le.value.strip(): return le.value.strip()
    return auto_label

def ov_on_plot(b):
    with ov_out_w:
        clear_output(wait=True)
        if not ov_configs:
            print('❌ Configure first'); return
        mode = ov_mode_sel.value
        do_trim = ov_trim_cb.value
        tmax = 20.0
        show_traces = ov_trace_sel.value == 'Traces'

        sg_tag = f' [SG w={ov_sg_global_win.value} p={ov_sg_global_poly.value}]' if ov_sg_global_cb.value else ''

        groups = OrderedDict()
        for tid in ov_selected_order:
            g = (ov_group_dropdowns.get(tid) or type('',(),{'value':''})()).value.strip() or f'_solo_{tid}'
            groups.setdefault(g, []).append(tid)

        fig, ax = plt.subplots(figsize=(11, 5.5), dpi=DPI)
        yl = 'Norm.' if mode == 'Normalized (0-1)' else 'ΔmV'

        for g, tids in groups.items():
            solo = g.startswith('_solo_')

            for tid in tids:
                col = _ov_trial_color(tid)
                if show_traces:
                    for x, y, auto_lbl in ov_build_traces(tid, mode, do_trim, tmax):
                        lbl = ov_get_legend(tid, auto_lbl)
                        ax.plot(x, y, color=col, lw=1.8, alpha=0.6, label=lbl)
                else:
                    r = ov_build_curve(tid, mode, do_trim, tmax)
                    if r is None: continue
                    x, y, auto_lbl, sel = r
                    if solo:
                        lbl = ov_get_legend(tid, auto_lbl)
                        ax.plot(x, y, color=col, lw=1.8, label=lbl)

            if not solo:
                grp_xs, grp_ys = [], []
                for tid in tids:
                    ch_curves = ov_build_channel_curves(tid, mode, do_trim, tmax)
                    grp_xs.extend([c[0] for c in ch_curves])
                    grp_ys.extend([c[1] for c in ch_curves])
                if grp_ys:
                    xc, yc = _ov_interp_avg(grp_xs, grp_ys)
                    if xc is not None:
                        le = ov_legend_edits.get(tids[0])
                        col = _ov_trial_color(tids[0])
                        if le and le.value.strip():
                            lbl = f'{le.value.strip()} (n={len(grp_ys)})'
                        else:
                            lbl = f'[{g}] AVG (n={len(grp_ys)})'
                        ax.plot(xc, yc, color=col, lw=1.8)

        if do_trim: ax.set_xlim(left=0)
        ax.set_xlim(right=tmax)
        ax.set_xlabel(ov_xlabel_w.value.strip() or 'Time (min)')
        ax.set_ylabel(ov_ylabel_w.value.strip() or yl)
        ax.grid(True, alpha=.4)
        ax.legend(fontsize=8, loc='best')

        if ov_title_w.value.strip():
            fig.suptitle(ov_title_w.value.strip(), fontsize=13, fontweight='bold')
        else:
            tags = (' [trimmed]' if do_trim else '') + sg_tag
            fig.suptitle(f'Overlay — {mode}{tags}', fontsize=13, fontweight='bold')
        plt.tight_layout(rect=[0, 0, 1, .93])
        _ov_save_fig(fig, 'overlay.png')
        plt.show()

        rows = []
        for g, tids in groups.items():
            gn = g if not g.startswith('_solo_') else '—'
            for tid in tids:
                r = ov_build_curve(tid, mode, do_trim, tmax)
                if r is None: continue
                _, y, _, sel = r
                rows.append(dict(Group=gn, Trial=tid, Channels=','.join(sel), N=len(y),
                    Start=f'{y.iloc[0]:.1f}', End=f'{y.iloc[-1]:.1f}',
                    **{'Δ': f'{y.iloc[-1]-y.iloc[0]:+.1f}'}, Max=f'{y.max():.1f}', Mean=f'{y.mean():.1f}'))
        if rows: display(pd.DataFrame(rows))

ov_plot_btn.on_click(ov_on_plot)

display(HTML(
    '<style>'
    '.widget-select select,.widget-select-multiple select{background:#1e1e1e!important;color:#f0f0f0!important;border:1px solid #555!important;font-size:14px!important}'
    '.widget-radio-box label,.widget-checkbox label,.widget-label{color:#ccc!important;font-size:14px!important}'
    '.widget-readout{color:#e0e0e0!important}.widget-text input{background:#1e1e1e!important;color:#f0f0f0!important;border:1px solid #555!important}'
    '.widget-dropdown select{background:#1e1e1e!important;color:#f0f0f0!important;border:1px solid #555!important}'
    '.dataframe th,.dataframe td{font-size:13px!important;color:#e0e0e0!important}'
    '.dataframe th{background:#2a2a2a!important}'
    '.dataframe tr:nth-child(even){background:#1a1a1a!important}'
    '.dataframe tr:nth-child(odd){background:#252525!important}'
    '</style>'))

display(widgets.HTML('<h2 style="color:#3498db;margin-bottom:4px">Overlay</h2>'))
display(widgets.HBox([ov_refresh_btn, ov_status]))
display(widgets.HTML('<hr style="border-color:#444">'))
display(ov_trial_sel)
display(ov_select_btn)
display(widgets.HTML('<hr style="border-color:#333">'))
display(ov_cfg_lbl)
display(ov_cfg_box)
display(widgets.HTML('<hr style="border-color:#333">'))
display(widgets.HBox([ov_mode_sel, widgets.VBox([ov_trace_sel, ov_trim_cb])]))
display(widgets.HBox([ov_sg_global_cb, ov_sg_global_win, ov_sg_global_poly]))
display(widgets.HTML('<hr style="border-color:#333;margin:4px 0">'))
display(ov_title_w)
display(widgets.HBox([ov_xlabel_w, ov_ylabel_w]))
display(widgets.HTML('<hr style="border-color:#333;margin:4px 0">'))
display(ov_plot_btn)
display(ov_out_w)
ov_refresh()

HTML(value='<h2 style="color:#3498db;margin-bottom:4px">Overlay</h2>')

HTML(value='<hr style="border-color:#444">')

SelectMultiple(description='Trials:', layout=Layout(height='180px', width='95%'), options=(('10^6 40 min run',…

Button(button_style='info', description='Select Trials → Configure', icon='arrow-right', layout=Layout(height=…

HTML(value='<hr style="border-color:#333">')

HTML(value='')

... (3 outputs left out)

Text(value='', description='Title:', layout=Layout(width='500px'), placeholder='auto-generated if blank', styl…

HTML(value='<hr style="border-color:#333;margin:4px 0">')

Button(button_style='primary', description='Generate Plot', icon='line-chart', layout=Layout(height='40px', wi…

Output()

This will allow you to overlay the individual CSV files' fluorescence curves, average them, and smooth with SG filter.

In [ ]:
# @title Tt
import numpy as np, pandas as pd, matplotlib.pyplot as plt, io, base64, ipywidgets as widgets
from IPython.display import display, clear_output, HTML as DHTML
from collections import OrderedDict
from scipy.stats import linregress
from matplotlib.ticker import MultipleLocator

DPI = 300
OVERLAY_LW = 1.8

def _save_fig(fig, filename='plot.png'):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=DPI, bbox_inches='tight')
    buf.seek(0)
    b64 = base64.b64encode(buf.read()).decode()
    display(DHTML(f'<a download="{filename}" href="data:image/png;base64,{b64}" '
                  f'style="color:#4fc3f7;font-size:12px">⬇ Download {filename}</a>'))

def _emp_deriv_tt(x, y, method='1st'):
    if len(y) < 7: return None, None
    imin = np.argmin(y)
    if imin >= len(y) - 3: return None, None
    x2, y2 = x[imin:], y[imin:]
    d1 = np.gradient(y2, x2)
    m = max(2, len(d1) // 20)
    if len(d1) <= 2 * m: return None, None
    if method == '1st':
        d1c, xc, y2c = d1[m:-m], x2[m:-m], y2[m:-m]
        pk = np.argmax(d1c)
        if d1c[pk] <= 0: return None, None
        return float(xc[pk]), float(y2c[pk])
    else:
        d2 = np.gradient(d1, x2)
        d2c, xc, d1c2, y2c = d2[m:-m], x2[m:-m], d1[m:-m], y2[m:-m]
        pk = np.argmax(d2c)
        if d2c[pk] <= 0: return None, None
        if d1c2[pk] <= 0: return None, None
        return float(xc[pk]), float(y2c[pk])

def _emp_first_crossing(x, y, level):
    for i in range(len(y) - 1):
        if (y[i] - level) * (y[i+1] - level) <= 0:
            dy = y[i+1] - y[i]
            if dy == 0: return float(x[i])
            return float(x[i] + (level - y[i]) / dy * (x[i+1] - x[i]))
    return None

def _emp_tt_with_threshold_error(x, y, threshold_mean, threshold_sd):
    tt_mean = _emp_first_crossing(x, y, threshold_mean)
    if threshold_sd > 0:
        tt_lo = _emp_first_crossing(x, y, threshold_mean - threshold_sd)
        tt_hi = _emp_first_crossing(x, y, threshold_mean + threshold_sd)
    else:
        tt_lo, tt_hi = tt_mean, tt_mean
    return tt_mean, tt_lo, tt_hi

def _emp_get_traces(tid, mode, tmax):
    c = ov_configs.get(tid)
    df = R.get(tid)
    if not c or df is None: return []
    sel = [k for k, v in c['ch'].items() if v.value]
    out = []
    for ch in sel:
        if ch not in df.columns: continue
        mk = df[ch].notna()
        x = _ov_get_time(df)[mk].reset_index(drop=True)
        y = df[ch][mk].reset_index(drop=True)
        y = _ov_maybe_smooth(y, tid)
        y, _ = _ov_xform(y, mode)
        x, y = _ov_trim_at_min(x, y)
        x, y = _ov_clip(x, y, tmax)
        if len(x) > 0:
            out.append((x.values, y.values, f'{tid}/{ch}'))
    return out

def _emp_get_display_traces(tid, mode, do_trim, tmax):
    c = ov_configs.get(tid)
    df = R.get(tid)
    if not c or df is None: return []
    sel = [k for k, v in c['ch'].items() if v.value]
    out = []
    for ch in sel:
        if ch not in df.columns: continue
        mk = df[ch].notna()
        x = _ov_get_time(df)[mk].reset_index(drop=True)
        y = df[ch][mk].reset_index(drop=True)
        y = _ov_maybe_smooth(y, tid)
        y, _ = _ov_xform(y, mode)
        if do_trim: x, y = _ov_trim_at_min(x, y)
        x, y = _ov_clip(x, y, tmax)
        if len(x) > 0:
            out.append((x.values, y.values, f'{tid}/{ch}'))
    return out

def _emp_build_groups():
    groups = OrderedDict()
    for tid in ov_selected_order:
        g = (ov_group_dropdowns.get(tid) or type('',(),{'value':''})()).value.strip() or f'_solo_{tid}'
        groups.setdefault(g, []).append(tid)
    return groups

def _emp_parse_conc(label):
    import re
    for p in [r'(\d+\.?\d*)\s*[eEx×]\s*([+-]?\d+)', r'10\^?\s*([+-]?\d+)']:
        m = re.search(p, label)
        if m:
            g = m.groups()
            try: return float(g[0]) * 10**float(g[1]) if len(g) == 2 else 10**float(g[0])
            except: pass
    return 0.0

emp_method = widgets.ToggleButtons(
    options=['1st derivative max (inflection)', '2nd derivative max (onset)'],
    value='1st derivative max (inflection)',
    description='Method:', style={'description_width':'65px', 'button_width':'220px'},
    layout=widgets.Layout(width='600px'))
emp_ref_dd = widgets.Dropdown(options=[], description='Reference:',
                               style={'description_width':'75px'},
                               layout=widgets.Layout(width='350px'))
emp_refresh_btn = widgets.Button(description='🔄 Refresh', button_style='warning',
                                  layout=widgets.Layout(width='100px'))
emp_title_w = widgets.Text(value='', placeholder='uses overlay title if blank',
                            description='Title:', style={'description_width': '50px'},
                            layout=widgets.Layout(width='400px'))
emp_conc_box = widgets.VBox([])
emp_conc_w = {}
emp_show_fit = widgets.Checkbox(value=True, description='Show best fit line', indent=False)
emp_fig_width = widgets.FloatSlider(value=7, min=4, max=14, step=0.5,
                                     description='Fig width:', style={'description_width':'70px'},
                                     layout=widgets.Layout(width='350px'))
emp_btn = widgets.Button(description='Compute Tt', button_style='success', icon='line-chart',
                          layout=widgets.Layout(width='160px', height='36px'))
emp_out = widgets.Output()

def _emp_refresh(b=None):
    if not ov_selected_order or not ov_configs:
        emp_ref_dd.options = []; return
    groups = _emp_build_groups()
    mode = ov_mode_sel.value
    tmax = 20.0
    opts = []
    for g, tids in groups.items():
        solo = g.startswith('_solo_')
        le = ov_legend_edits.get(tids[0])
        label = le.value.strip() if le and le.value.strip() else (
            _ov_tname(tids[0]) if solo else f'[{g}]')
        n_ch = sum(len(_emp_get_traces(tid, mode, tmax)) for tid in tids)
        opts.append((f'{label}  ({n_ch} ch)', g))
    emp_ref_dd.options = opts
    if opts: emp_ref_dd.value = opts[0][1]

emp_refresh_btn.on_click(_emp_refresh)

def _emp_build_conc_cards():
    groups = _emp_build_groups()
    if not groups: emp_conc_box.children = []; return
    cards = []
    for g, tids in groups.items():
        solo = g.startswith('_solo_')
        le = ov_legend_edits.get(tids[0])
        ln = le.value.strip() if le and le.value.strip() else ''
        name = _ov_tname(tids[0]) if solo else f'[{g}]'
        if g not in emp_conc_w:
            emp_conc_w[g] = {
                'l': widgets.Text(value=ln, placeholder='e.g. 10^-3', description='Label:',
                                   style={'description_width':'40px'}, layout=widgets.Layout(width='180px')),
                'n': widgets.Checkbox(value=False, description='NTC', indent=False,
                                       layout=widgets.Layout(width='60px')),
                'f': widgets.Checkbox(value=True, description='Fit', indent=False,
                                       layout=widgets.Layout(width='55px')),
            }
        else:
            w = emp_conc_w[g]
            if not w['l'].value.strip() and ln:
                w['l'].value = ln
        if emp_conc_w[g]['l'].value.strip().upper() == 'NTC':
            emp_conc_w[g]['n'].value = True
        lv = emp_conc_w[g]['l'].value.strip().lower()
        if lv in ('ntc', 'water'):
            emp_conc_w[g]['f'].value = False
        w = emp_conc_w[g]
        cards.append(widgets.HBox([
            widgets.HTML(f'<span style="color:#e0e0e0;font-size:12px;width:200px;display:inline-block">{name}</span>'),
            w['l'], w['n'], w['f'],
        ], layout=widgets.Layout(align_items='center', margin='1px 0')))
    for k in [k for k in emp_conc_w if k not in groups]: del emp_conc_w[k]
    emp_conc_box.children = [
        widgets.HTML('<p style="color:#888;font-size:11px;margin:2px 0">'
                     'Label = x-axis tick (concentration parsed from it). NTC = exclude from x-axis. '
                     'Fit = include in regression.</p>')
    ] + cards

def _emp_plot(b):
    with emp_out:
        clear_output(wait=True)
        if not ov_selected_order or not ov_configs:
            print('❌ Configure overlay first.'); return
        if not emp_ref_dd.value:
            print('❌ Select a reference group.'); return

        _emp_build_conc_cards()

        mode = ov_mode_sel.value
        do_trim_display = ov_trim_cb.value
        tmax = 20.0
        deriv_method = '1st' if '1st' in emp_method.value else '2nd'
        method_name = 'inflection (1st deriv max)' if deriv_method == '1st' else 'onset (2nd deriv max)'
        plot_title = emp_title_w.value.strip() or ''
        fig_w = emp_fig_width.value

        groups = _emp_build_groups()

        group_data = OrderedDict()
        group_display = OrderedDict()
        for g, tids in groups.items():
            solo = g.startswith('_solo_')
            le = ov_legend_edits.get(tids[0])
            label = le.value.strip() if le and le.value.strip() else (
                _ov_tname(tids[0]) if solo else f'[{g}]')
            calc_traces, disp_traces = [], []
            for tid in tids:
                calc_traces.extend(_emp_get_traces(tid, mode, tmax))
                disp_traces.extend(_emp_get_display_traces(tid, mode, do_trim_display, tmax))
            if calc_traces:
                group_data[g] = {'label': label, 'traces': calc_traces}
                group_display[g] = {'label': label, 'traces': disp_traces}

        if not group_data:
            print('❌ No curves.'); return

        ref_key = emp_ref_dd.value
        if ref_key not in group_data:
            print('❌ Reference group not found.'); return

        ref = group_data[ref_key]
        deriv_y_vals, deriv_details = [], []
        for x, y, name in ref['traces']:
            tt, yval = _emp_deriv_tt(x, y, deriv_method)
            if tt is not None and yval is not None:
                deriv_y_vals.append(yval)
                deriv_details.append((name, tt, yval))

        if not deriv_y_vals:
            print(f'❌ Could not find {method_name} on {ref["label"]}.'); return

        threshold = np.mean(deriv_y_vals)
        threshold_sd = np.std(deriv_y_vals, ddof=1) if len(deriv_y_vals) > 1 else 0

        if mode == 'Normalized (0-1)':
            print(f'Reference: {ref["label"]}  |  {method_name}')
            print(f'   {threshold*100:.1f}% ± {threshold_sd*100:.1f}% (n={len(deriv_y_vals)})')
            thresh_label = f'{threshold:.1%}'
        else:
            print(f'Reference: {ref["label"]}  |  {method_name}')
            print(f'   y = {threshold:.1f} ± {threshold_sd:.1f} mV (n={len(deriv_y_vals)})')
            thresh_label = f'{threshold:.1f}'
        print(f'   Threshold = {threshold:.4f} ± {threshold_sd:.4f}\n')

        summary, all_detail = [], []
        for g, d in group_data.items():
            tts_mean, tts_lo, tts_hi = [], [], []
            for x, y, name in d['traces']:
                tt_m, tt_l, tt_h = _emp_tt_with_threshold_error(x, y, threshold, threshold_sd)
                tts_mean.append(tt_m)
                tts_lo.append(tt_l)
                tts_hi.append(tt_h)

                all_detail.append({'Group': d['label'], 'Trace': name,
                                   'Tt (min)': f'{tt_m:.2f}' if tt_m is not None else 'DNF',
                                   'Tt @ thresh−SD': f'{tt_l:.2f}' if tt_l is not None else 'DNF',
                                   'Tt @ thresh+SD': f'{tt_h:.2f}' if tt_h is not None else 'DNF'})

            valid_mean = [t for t in tts_mean if t is not None]
            valid_lo = [t for t in tts_lo if t is not None]
            valid_hi = [t for t in tts_hi if t is not None]

            if valid_mean:
                mean_tt = np.mean(valid_mean)
                replicate_sd = np.std(valid_mean, ddof=1) if len(valid_mean) > 1 else 0.0
                mean_tt_lo = np.mean(valid_lo) if valid_lo else mean_tt
                mean_tt_hi = np.mean(valid_hi) if valid_hi else mean_tt
                thresh_shift_lo = abs(mean_tt - mean_tt_lo)
                thresh_shift_hi = abs(mean_tt_hi - mean_tt)
                combined_sd_lo = np.sqrt(replicate_sd**2 + thresh_shift_lo**2)
                combined_sd_hi = np.sqrt(replicate_sd**2 + thresh_shift_hi**2)
            else:
                mean_tt = None
                replicate_sd = 0.0
                combined_sd_lo, combined_sd_hi = 0.0, 0.0
                thresh_shift_lo, thresh_shift_hi = 0.0, 0.0

            summary.append((d['label'], mean_tt, replicate_sd,
                            combined_sd_lo, combined_sd_hi,
                            thresh_shift_lo, thresh_shift_hi,
                            len(valid_mean), len(tts_mean), valid_mean, g))

        summary.sort(key=lambda s: s[1] if s[1] is not None else 999)
        colors = [plt.cm.tab10(i % 10) for i in range(len(summary))]
        thresh_text = f'{thresh_label} ± {threshold_sd:.1f} mV from {ref["label"]} {method_name}'

        fig1, ax1 = plt.subplots(figsize=(fig_w, 5), dpi=DPI)
        ci = 0
        for g in groups:
            if g not in group_display: continue
            d = group_display[g]
            col = plt.cm.tab10(ci % 10); ci += 1
            gxs = [x for x, _, _ in d['traces']]
            gys = [y for _, y, _ in d['traces']]
            if gys:
                x0, x1 = max(a[0] for a in gxs), min(a[-1] for a in gxs)
                if x1 > x0:
                    xc = np.linspace(x0, x1, min(len(a) for a in gys))
                    yc = np.mean([np.interp(xc, xi, yi) for xi, yi in zip(gxs, gys)], axis=0)
                    ax1.plot(xc, yc, color=col, lw=OVERLAY_LW, label=d['label'])
                    avg_tt = _emp_first_crossing(xc, yc, threshold)
                    if avg_tt is not None:
                        ax1.plot(avg_tt, threshold, 'o', color=col, ms=7,
                                  markeredgecolor='white', markeredgewidth=1.0, zorder=6)

        ax1.axhline(threshold, color='#ff4444', ls='--', lw=1.2, alpha=0.85, zorder=5)
        if threshold_sd > 0:
            ax1.axhspan(threshold - threshold_sd, threshold + threshold_sd,
                        color='#ff4444', alpha=0.08, zorder=1)
            ax1.axhline(threshold - threshold_sd, color='#ff4444', ls=':', lw=0.5, alpha=0.5, zorder=4)
            ax1.axhline(threshold + threshold_sd, color='#ff4444', ls=':', lw=0.5, alpha=0.5, zorder=4)
        ax1.annotate(f'Threshold = {thresh_label} ± {threshold_sd:.1f}',
                     xy=(0.98, threshold), xycoords=('axes fraction', 'data'),
                     xytext=(0, -12), textcoords='offset points',
                     fontsize=8, color='#ff4444', fontweight='bold',
                     va='top', ha='right',
                     bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#ff4444', alpha=0.9),
                     zorder=7)
        ax1.set_xlabel('Time (min)', fontsize=11)
        ax1.set_ylabel('Norm.' if mode == 'Normalized (0-1)' else 'ΔmV', fontsize=11)
        ax1.set_title(plot_title or 'Fluorescence Curves with Empirical Threshold',
                      fontsize=12, fontweight='bold')
        ax1.grid(True, ls='--', alpha=0.3); ax1.legend(fontsize=8, loc='best')
        ax1.set_xlim(right=tmax)
        plt.tight_layout()
        _save_fig(fig1, 'tt_overlay.png')
        plt.show(); plt.close(fig1)

        if emp_conc_w:
            ntc_tts_data, ntc_tts_lo, ntc_tts_hi, samples = [], [], [], []
            for g, d in group_data.items():
                w = emp_conc_w.get(g)
                if not w: continue
                label, is_ntc = w['l'].value.strip(), w['n'].value
                use_fit = w['f'].value

                tts_m, tts_l, tts_h = [], [], []
                for x, y, _ in d['traces']:
                    tm, tl, th = _emp_tt_with_threshold_error(x, y, threshold, threshold_sd)
                    tts_m.append(tm)
                    tts_l.append(tl)
                    tts_h.append(th)

                valid_m = [t for t in tts_m if t is not None]
                valid_l = [t for t in tts_l if t is not None]
                valid_h = [t for t in tts_h if t is not None]

                if is_ntc:
                    ntc_tts_data.extend(valid_m)
                    ntc_tts_lo.extend(valid_l)
                    ntc_tts_hi.extend(valid_h)
                else:
                    conc = _emp_parse_conc(label)
                    if conc <= 0: continue
                    log_c = np.log10(conc)
                    label = label or f'{conc:.1e}'

                    if valid_m:
                        mean = np.mean(valid_m)
                        rep_sd = np.std(valid_m, ddof=1) if len(valid_m) > 1 else 0.0
                        mean_lo = np.mean(valid_l) if valid_l else mean
                        mean_hi = np.mean(valid_h) if valid_h else mean
                        tsh_lo = abs(mean - mean_lo)
                        tsh_hi = abs(mean_hi - mean)
                        csd_lo = np.sqrt(rep_sd**2 + tsh_lo**2)
                        csd_hi = np.sqrt(rep_sd**2 + tsh_hi**2)
                    else:
                        mean = None
                        rep_sd, csd_lo, csd_hi = 0.0, 0.0, 0.0

                    samples.append((log_c, mean, rep_sd, csd_lo, csd_hi,
                                    len(valid_m), len(tts_m), label, valid_m, use_fit))

            samples.sort(key=lambda s: s[0])

            if samples:
                fig3, ax3 = plt.subplots(figsize=(fig_w, 5), dpi=DPI)

                all_tick_positions = []
                all_tick_labels = []

                xp_lin = [10**s[0] for s in samples if s[1] is not None]
                yp = [s[1] for s in samples if s[1] is not None]
                ep_lo = [s[3] for s in samples if s[1] is not None]
                ep_hi = [s[4] for s in samples if s[1] is not None]

                if xp_lin:
                    ax3.errorbar(xp_lin, yp, yerr=[ep_lo, ep_hi], fmt='o', color='black',
                                  capsize=3, capthick=0.8, lw=0.8, ms=5, zorder=5,
                                  label='Mean Tt ± combined SD')


                for s in samples:
                    all_tick_positions.append(10**s[0])
                    all_tick_labels.append(s[7])

                mt = max(yp) * 1.15 if yp else 20
                for lc, mean, rep_sd, csd_lo, csd_hi, nd, nt, lbl, tts, _ in samples:
                    dnf = nt - nd
                    if dnf > 0:
                        ax3.plot(10**lc, mt, 'v', color='#cc0000', ms=6, zorder=6)
                        ax3.annotate(f'{dnf} DNF', (10**lc, mt), textcoords='offset points',
                                     xytext=(6, 2), fontsize=7, color='#cc0000')

                if ntc_tts_data:
                    nm = np.mean(ntc_tts_data)
                    ntc_rep_sd = np.std(ntc_tts_data, ddof=1) if len(ntc_tts_data) > 1 else 0.0
                    ntc_mean_lo = np.mean(ntc_tts_lo) if ntc_tts_lo else nm
                    ntc_mean_hi = np.mean(ntc_tts_hi) if ntc_tts_hi else nm
                    ntc_tsh_lo = abs(nm - ntc_mean_lo)
                    ntc_tsh_hi = abs(ntc_mean_hi - nm)
                    ntc_csd_lo = np.sqrt(ntc_rep_sd**2 + ntc_tsh_lo**2)
                    ntc_csd_hi = np.sqrt(ntc_rep_sd**2 + ntc_tsh_hi**2)

                    if all_tick_positions:
                        ntc_x = 10**(np.log10(min(all_tick_positions)) - 1.0)
                    else:
                        ntc_x = 1e-12

                    ax3.errorbar(ntc_x, nm, yerr=[[ntc_csd_lo], [ntc_csd_hi]], fmt='s',
                                  color='#cc0000', capsize=3, capthick=0.8, lw=0.8, ms=6,
                                  markeredgecolor='white', markeredgewidth=0.5, zorder=5,
                                  label=f'NTC = {nm:.1f} ± {ntc_rep_sd:.1f} min (n={len(ntc_tts_data)})')

                    all_tick_positions.insert(0, ntc_x)
                    all_tick_labels.insert(0, 'NTC')

                fit_txt = ''
                if emp_show_fit.value:
                    xf = [s[0] for s in samples if s[1] is not None and s[9]]
                    yf = [s[1] for s in samples if s[1] is not None and s[9]]
                    if len(xf) >= 2:
                        sl, ic, rv, pv, se = linregress(np.array(xf), np.array(yf))
                        xl_log = np.linspace(min(xf) - 0.3, max(xf) + 0.3, 100)
                        xl_lin = 10**xl_log
                        ax3.plot(xl_lin, sl * xl_log + ic, '-', color='#333', lw=0.8, alpha=0.6,
                                  label=f'y = {sl:.2f}·log₁₀(x) + {ic:.2f}  (R² = {rv**2:.3f})')
                        fit_txt = f'Fit: Tt = {sl:.2f}·log₁₀(conc) + {ic:.2f}  |  R² = {rv**2:.4f}  |  p = {pv:.2e}'
                        excluded = [s[7] for s in samples if s[1] is not None and not s[9]]
                        if excluded:
                            fit_txt += f'\n  Excluded from fit: {", ".join(excluded)}'

                ax3.set_xscale('log')
                ax3.set_xlabel('Initial Target DNA Concentration (µg/µL)', fontsize=11)
                ax3.set_ylabel('Tt (min)', fontsize=11)
                ax3.yaxis.set_major_locator(MultipleLocator(1))
                ax3.set_title(
                    plot_title or 'Tt as a Function of Initial Target DNA Concentration',
                    fontsize=12, fontweight='bold')
                ax3.set_xticks(all_tick_positions)
                ax3.set_xticklabels(all_tick_labels, fontsize=9, rotation=30, ha='right')
                ax3.margins(x=0.08); ax3.set_ylim(bottom=0)
                ax3.grid(True, which='major', ls='--', alpha=0.25)
                ax3.grid(True, which='minor', ls=':', alpha=0.12)
                ax3.legend(fontsize=8, loc='lower left', bbox_to_anchor=(0.04, 0.04), framealpha=0.9)
                plt.tight_layout()
                _save_fig(fig3, 'tt_standard_curve.png')
                plt.show(); plt.close(fig3)

                if fit_txt: print(fit_txt)
                if ntc_tts_data:
                    print(f'NTC: {np.mean(ntc_tts_data):.2f} ± '
                          f'{np.std(ntc_tts_data, ddof=1) if len(ntc_tts_data)>1 else 0:.2f} min '
                          f'(n={len(ntc_tts_data)})')
                print(f'\nThreshold uncertainty: {threshold:.2f} ± {threshold_sd:.2f} mV '
                      f'(n={len(deriv_y_vals)} reference traces)')

                std_rows = []
                for lc, mean, rep_sd, csd_lo, csd_hi, nd, nt, lbl, tts, uf in samples:
                    cv = (rep_sd/mean*100) if mean and len(tts) > 1 else 0
                    std_rows.append({
                        'Conc': lbl, 'log₁₀': f'{lc:.2f}',
                        'Tt': f'{mean:.2f}' if mean else 'DNF',
                        'Rep SD': f'{rep_sd:.3f}' if len(tts) > 1 else '—',
                        'Comb −SD': f'{csd_lo:.3f}',
                        'Comb +SD': f'{csd_hi:.3f}',
                        'n': f'{nd}/{nt}', 'CV%': f'{cv:.1f}' if cv else '—',
                        'In fit': '✓' if uf else '✗'})
                if ntc_tts_data:
                    nm = np.mean(ntc_tts_data)
                    ns = np.std(ntc_tts_data, ddof=1) if len(ntc_tts_data) > 1 else 0
                    std_rows.append({
                        'Conc': 'NTC', 'log₁₀': '—', 'Tt': f'{nm:.2f}',
                        'Rep SD': f'{ns:.3f}' if len(ntc_tts_data) > 1 else '—',
                        'Comb −SD': f'{ntc_csd_lo:.3f}',
                        'Comb +SD': f'{ntc_csd_hi:.3f}',
                        'n': str(len(ntc_tts_data)),
                        'CV%': f'{ns/nm*100:.1f}' if nm and len(ntc_tts_data) > 1 else '—',
                        'In fit': '—'})
                display(pd.DataFrame(std_rows))

        tbl = []
        for (lbl, mean, rep_sd, csd_lo, csd_hi, tsh_lo, tsh_hi,
             n_det, n_tot, tts, _) in summary:
            cv = (rep_sd / mean * 100) if mean and len(tts) > 1 else 0
            tbl.append({
                'Group': lbl,
                'Mean Tt': f'{mean:.2f}' if mean else 'DNF',
                'Rep SD': f'{rep_sd:.3f}' if len(tts) > 1 else '—',
                'Thresh shift': f'−{tsh_lo:.3f}/+{tsh_hi:.3f}',
                'Comb −SD': f'{csd_lo:.3f}',
                'Comb +SD': f'{csd_hi:.3f}',
                'n': f'{n_det}/{n_tot}',
                'CV%': f'{cv:.1f}' if cv else '—'})
        valid_means = [s[1] for s in summary if s[1] is not None]
        if len(valid_means) >= 2:
            tbl.append({'Group': 'Spread',
                        'Mean Tt': f'{max(valid_means)-min(valid_means):.2f}',
                        'Rep SD': '', 'Thresh shift': '', 'Comb −SD': '', 'Comb +SD': '',
                        'n': '', 'CV%': ''})
        display(pd.DataFrame(tbl))

        if all_detail:
            display(DHTML('<details><summary style="cursor:pointer;color:#888;font-size:12px">'
                          'Per-channel detail (with threshold uncertainty)</summary>'))
            display(pd.DataFrame(all_detail))
            display(DHTML('</details>'))

emp_btn.on_click(_emp_plot)
_emp_refresh()

display(widgets.VBox([
    widgets.HTML('<h3 style="color:#e65100;margin-bottom:4px">Tt — Empirical Threshold</h3>'
                 '<span style="color:#888;font-size:12px">1st deriv = steepest rise. 2nd deriv = onset. '
                 'Error bars combine replicate SD and threshold uncertainty in quadrature.</span>'),
    emp_method,
    widgets.HBox([emp_ref_dd, emp_refresh_btn]),
    emp_title_w,
    widgets.HTML('<hr style="border-color:#444;margin:8px 0">'),
    widgets.HTML('<b style="color:#4fc3f7;font-size:13px">Standard Curve</b>'),
    emp_show_fit,
    emp_fig_width,
    emp_conc_box,
    widgets.HTML('<hr style="border-color:#444;margin:8px 0">'),
    emp_btn,
    emp_out,
], layout=widgets.Layout(border='2px solid #e65100', padding='8px', margin='8px 0', border_radius='8px')))

Time to threshold plotter will display the groups selected in the previous Overlay block. It plots a threshold line on the fluorescence curves and also plots Tt as a function of concentration.

In [ ]:
# @title Inflection Point Traces Viewer
import numpy as np, matplotlib.pyplot as plt, io, base64, ipywidgets as widgets
from IPython.display import display, clear_output, HTML as DHTML

_inf_DPI = 300
_inf_LW = 1.2
_inf_TRACE_COLOR = '#1976D2'
_inf_DOT_COLOR = '#D32F2F'

def _inf_save_fig(fig, filename='plot.png'):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=_inf_DPI, bbox_inches='tight')
    buf.seek(0)
    b64 = base64.b64encode(buf.read()).decode()
    display(DHTML(f'<a download="{filename}" href="data:image/png;base64,{b64}" '
                  f'style="color:#4fc3f7;font-size:12px">⬇ Download {filename}</a>'))

def _inf_format_label_superscript(label):
    import re
    def _replace(m):
        base = m.group(1)
        exp = m.group(2)
        return f'{base}$^{{{exp}}}$'
    return re.sub(r'(\d+)\^([+-]?\d+)', _replace, label)

inf_group_dd = widgets.Dropdown(options=[], description='Group:',
                                 style={'description_width':'55px'},
                                 layout=widgets.Layout(width='400px'))
inf_method_dd = widgets.ToggleButtons(
    options=['1st derivative max (inflection)', '2nd derivative max (onset)'],
    value='1st derivative max (inflection)',
    description='Method:', style={'description_width':'65px', 'button_width':'220px'},
    layout=widgets.Layout(width='600px'))
inf_title_w = widgets.Text(value='', placeholder='auto-generated if blank',
                            description='Title:', style={'description_width':'50px'},
                            layout=widgets.Layout(width='400px'))
inf_refresh_btn = widgets.Button(description='🔄 Refresh', button_style='warning',
                                  layout=widgets.Layout(width='100px'))
inf_btn = widgets.Button(description='Plot Traces + Inflections', button_style='info',
                          icon='search', layout=widgets.Layout(width='220px', height='36px'))
inf_out = widgets.Output()

def _inf_refresh(b=None):
    if not ov_selected_order or not ov_configs:
        inf_group_dd.options = []; return
    groups = _emp_build_groups()
    mode = ov_mode_sel.value
    tmax = 20.0
    opts = []
    for g, tids in groups.items():
        solo = g.startswith('_solo_')
        le = ov_legend_edits.get(tids[0])
        label = le.value.strip() if le and le.value.strip() else (
            _ov_tname(tids[0]) if solo else f'[{g}]')
        n_ch = sum(len(_emp_get_traces(tid, mode, tmax)) for tid in tids)
        opts.append((f'{label}  ({n_ch} traces)', g))
    inf_group_dd.options = opts
    if opts: inf_group_dd.value = opts[0][1]

inf_refresh_btn.on_click(_inf_refresh)

def _inf_plot(b):
    with inf_out:
        clear_output(wait=True)
        if not ov_selected_order or not ov_configs:
            print('❌ Configure overlay first.'); return
        if not inf_group_dd.value:
            print('❌ Select a group.'); return

        mode = ov_mode_sel.value
        do_trim = ov_trim_cb.value
        tmax = 20.0
        deriv_method = '1st' if '1st' in inf_method_dd.value else '2nd'
        method_label = 'inflection (1st deriv max)' if deriv_method == '1st' else 'onset (2nd deriv max)'

        groups = _emp_build_groups()
        g = inf_group_dd.value
        if g not in groups:
            print('❌ Group not found.'); return

        tids = groups[g]
        solo = g.startswith('_solo_')
        le = ov_legend_edits.get(tids[0])
        group_label_raw = le.value.strip() if le and le.value.strip() else (
            _ov_tname(tids[0]) if solo else f'[{g}]')

        display_traces = []
        calc_traces = []
        for tid in tids:
            display_traces.extend(_emp_get_display_traces(tid, mode, do_trim, tmax))
            calc_traces.extend(_emp_get_traces(tid, mode, tmax))

        if not display_traces:
            print('❌ No traces found for this group.'); return

        calc_by_name = {name: (x, y) for x, y, name in calc_traces}

        inflection_data = []
        for x_disp, y_disp, name in display_traces:
            if name in calc_by_name:
                x_calc, y_calc = calc_by_name[name]
                tt_x, tt_y_calc = _emp_deriv_tt(x_calc, y_calc, deriv_method)
            else:
                tt_x, tt_y_calc = None, None

            if tt_x is not None:
                tt_y_disp = float(np.interp(tt_x, x_disp, y_disp))
                inflection_data.append((tt_x, tt_y_disp, name))
            else:
                inflection_data.append((None, None, name))

        n_traces = len(display_traces)
        n_detected = sum(1 for d in inflection_data if d[0] is not None)

        fig, ax = plt.subplots(figsize=(11, 5), dpi=_inf_DPI)

        # Plot all traces in the same color
        for i, (x_disp, y_disp, name) in enumerate(display_traces):
            lbl = f'Replicates (n = {n_traces})' if i == 0 else None
            ax.plot(x_disp, y_disp, color=_inf_TRACE_COLOR, lw=_inf_LW, alpha=0.7, label=lbl)

        # Plot inflection dots
        first_dot = True
        for i, (tt_x, tt_y, name) in enumerate(inflection_data):
            if tt_x is not None:
                lbl = 'Inflection point' if first_dot else None
                ax.plot(tt_x, tt_y, 'o', color=_inf_DOT_COLOR, ms=8,
                        markeredgecolor='white', markeredgewidth=1.2, zorder=6, label=lbl)
                first_dot = False

        # Mean ± SD horizontal band
        valid_inf_y = [d[1] for d in inflection_data if d[1] is not None]
        valid_inf_x = [d[0] for d in inflection_data if d[0] is not None]

        if valid_inf_y:
            mean_y = np.mean(valid_inf_y)
            sd_y = np.std(valid_inf_y, ddof=1) if len(valid_inf_y) > 1 else 0
            ax.axhline(mean_y, color='#ff4444', ls='--', lw=1.0, alpha=0.7, zorder=4)
            if sd_y > 0:
                ax.axhspan(mean_y - sd_y, mean_y + sd_y,
                           color='#ff4444', alpha=0.08, zorder=1)
                ax.axhline(mean_y - sd_y, color='#ff4444', ls=':', lw=0.5, alpha=0.5, zorder=4)
                ax.axhline(mean_y + sd_y, color='#ff4444', ls=':', lw=0.5, alpha=0.5, zorder=4)

            if mode == 'Normalized (0-1)':
                thresh_str = f'{mean_y:.1%} ± {sd_y:.1%}'
            else:
                thresh_str = f'{mean_y:.1f} ± {sd_y:.1f} mV'

            ax.annotate(f'Mean ± SD\ny = {thresh_str}',
                        xy=(0.98, mean_y), xycoords=('axes fraction', 'data'),
                        xytext=(0, -12), textcoords='offset points',
                        fontsize=7.5, color='#ff4444', fontweight='bold',
                        va='top', ha='right',
                        bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#ff4444', alpha=0.9),
                        zorder=7)

        ax.set_xlabel('Time (min)', fontsize=11)
        ax.set_ylabel('Norm.' if mode == 'Normalized (0-1)' else 'ΔmV', fontsize=11)

        if inf_title_w.value.strip():
            title = _inf_format_label_superscript(inf_title_w.value.strip())
        else:
            formatted_group = _inf_format_label_superscript(group_label_raw)
            title = f'Inflection Points of {formatted_group} Replicates'

        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.grid(True, ls='--', alpha=0.3)
        ax.set_xlim(right=tmax)
        ax.legend(fontsize=8, loc='upper left', framealpha=0.9)

        plt.tight_layout()
        safe_fname = group_label_raw.replace(' ', '_').replace('/', '_').replace('^', '')
        _inf_save_fig(fig, f'inflection_traces_{safe_fname}.png')
        plt.show(); plt.close(fig)

        # ── Summary table ──
        print(f'\n{method_label.upper()} for {group_label_raw}')
        print(f'{"Trace":<35} {"Tt (min)":>10}  {"y @ inflection":>14}')
        print('─' * 65)
        for tt_x, tt_y, name in inflection_data:
            short = name.split('/')[-1] if '/' in name else name
            if tt_x is not None:
                y_str = f'{tt_y:.4f}' if mode == 'Normalized (0-1)' else f'{tt_y:.1f}'
                print(f'{short:<35} {tt_x:>10.2f}  {y_str:>14}')
            else:
                print(f'{short:<35} {"DNF":>10}  {"—":>14}')
        if valid_inf_x:
            mean_x = np.mean(valid_inf_x)
            sd_x = np.std(valid_inf_x, ddof=1) if len(valid_inf_x) > 1 else 0
            print('─' * 65)
            print(f'{"Mean ± SD":<35} {mean_x:>10.2f} ± {sd_x:.2f}  '
                  f'(n={len(valid_inf_x)}/{len(inflection_data)})')

inf_btn.on_click(_inf_plot)
_inf_refresh()

display(widgets.VBox([
    widgets.HTML('<h3 style="color:#1565c0;margin-bottom:4px">Individual Traces — Inflection Points</h3>'
                 '<span style="color:#888;font-size:12px">Shows each trace in the selected group '
                 'with its derivative-based inflection/onset point marked. '
                 'Mean ± SD band shows the spread of inflection y-values across replicates.</span>'),
    inf_method_dd,
    widgets.HBox([inf_group_dd, inf_refresh_btn]),
    inf_title_w,
    inf_btn,
    inf_out,
], layout=widgets.Layout(border='2px solid #1565c0', padding='8px', margin='8px 0', border_radius='8px')))

In [ ]:
# @title All Groups — Threshold Crossing View
import numpy as np, matplotlib.pyplot as plt, io, base64, ipywidgets as widgets
from IPython.display import display, clear_output, HTML as DHTML
from collections import OrderedDict

_ac_DPI = 300
_ac_LW = 1.2

_ac_GROUP_COLORS = [
    '#1565C0',  # blue
    '#EF6C00',  # orange
    '#2E7D32',  # green
    '#D32F2F',  # red
    '#7B1FA2',  # purple
    '#6D4C41',  # brown
    '#F06292',  # pink
    '#757575',  # gray
]

def _ac_save_fig(fig, filename='plot.png'):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=_ac_DPI, bbox_inches='tight')
    buf.seek(0)
    b64 = base64.b64encode(buf.read()).decode()
    display(DHTML(f'<a download="{filename}" href="data:image/png;base64,{b64}" '
                  f'style="color:#4fc3f7;font-size:12px">⬇ Download {filename}</a>'))

def _ac_format_label_superscript(label):
    import re
    def _replace(m):
        return f'{m.group(1)}$^{{{m.group(2)}}}$'
    return re.sub(r'(\d+)\^([+-]?\d+)', _replace, label)

# ── Widgets ──
ac_ref_dd = widgets.Dropdown(options=[], description='Threshold from:',
                              style={'description_width':'100px'},
                              layout=widgets.Layout(width='450px'))
ac_show_dd = widgets.SelectMultiple(options=[], description='Show groups:',
                                     style={'description_width':'100px'},
                                     layout=widgets.Layout(width='450px', height='140px'))
ac_title_w = widgets.Text(value='', placeholder='auto-generated if blank',
                           description='Title:', style={'description_width':'50px'},
                           layout=widgets.Layout(width='400px'))
ac_refresh_btn = widgets.Button(description='🔄 Refresh', button_style='warning',
                                 layout=widgets.Layout(width='100px'))
ac_select_all_btn = widgets.Button(description='Select All', button_style='info',
                                    layout=widgets.Layout(width='100px'))
ac_btn = widgets.Button(description='Plot', button_style='success',
                         icon='line-chart', layout=widgets.Layout(width='160px', height='36px'))
ac_out = widgets.Output()

def _ac_get_group_opts():
    if not ov_selected_order or not ov_configs:
        return []
    groups = _emp_build_groups()
    mode = ov_mode_sel.value
    tmax = 20.0
    opts = []
    for g, tids in groups.items():
        solo = g.startswith('_solo_')
        le = ov_legend_edits.get(tids[0])
        label = le.value.strip() if le and le.value.strip() else (
            _ov_tname(tids[0]) if solo else f'[{g}]')
        n_ch = sum(len(_emp_get_traces(tid, mode, tmax)) for tid in tids)
        opts.append((f'{label}  ({n_ch} traces)', g))
    return opts

def _ac_refresh(b=None):
    opts = _ac_get_group_opts()
    ac_ref_dd.options = opts
    ac_show_dd.options = opts
    if opts:
        ac_ref_dd.value = opts[0][1]
        ac_show_dd.value = [o[1] for o in opts]

def _ac_select_all(b=None):
    opts = ac_show_dd.options
    if opts:
        ac_show_dd.value = [o[1] if isinstance(o, tuple) else o for o in opts]

ac_refresh_btn.on_click(_ac_refresh)
ac_select_all_btn.on_click(_ac_select_all)

def _ac_plot(b):
    with ac_out:
        clear_output(wait=True)
        if not ov_selected_order or not ov_configs:
            print('❌ Configure overlay first.'); return
        if not ac_ref_dd.value:
            print('❌ Select a reference group for threshold.'); return
        if not ac_show_dd.value:
            print('❌ Select at least one group to show.'); return

        mode = ov_mode_sel.value
        do_trim = ov_trim_cb.value
        tmax = 20.0
        deriv_method = '1st'

        groups = _emp_build_groups()
        ref_key = ac_ref_dd.value
        selected_keys = list(ac_show_dd.value)

        if ref_key not in groups:
            print('❌ Reference group not found.'); return

        # ── Compute inflection threshold from reference group ──
        ref_tids = groups[ref_key]
        ref_calc_traces = []
        for tid in ref_tids:
            ref_calc_traces.extend(_emp_get_traces(tid, mode, tmax))

        if not ref_calc_traces:
            print('❌ No traces in reference group.'); return

        inf_y_vals = []
        for x, y, name in ref_calc_traces:
            tt_x, tt_y = _emp_deriv_tt(x, y, deriv_method)
            if tt_x is not None and tt_y is not None:
                inf_y_vals.append(tt_y)

        if not inf_y_vals:
            print('❌ Could not find inflection points in reference group.'); return

        threshold = np.mean(inf_y_vals)
        threshold_sd = np.std(inf_y_vals, ddof=1) if len(inf_y_vals) > 1 else 0

        ref_solo = ref_key.startswith('_solo_')
        ref_le = ov_legend_edits.get(ref_tids[0])
        ref_label_raw = ref_le.value.strip() if ref_le and ref_le.value.strip() else (
            _ov_tname(ref_tids[0]) if ref_solo else f'[{ref_key}]')
        ref_label_fmt = _ac_format_label_superscript(ref_label_raw)

        if mode == 'Normalized (0-1)':
            thresh_str = f'{threshold:.1%} ± {threshold_sd:.1%}'
        else:
            thresh_str = f'{threshold:.1f} ± {threshold_sd:.1f} mV'

        print(f'Reference: {ref_label_raw}')
        print(f'Inflection Mean ± SD: y = {thresh_str}  (n = {len(inf_y_vals)})\n')

        # ── Stable color map ──
        all_group_keys = list(groups.keys())
        color_map = {g: _ac_GROUP_COLORS[i % len(_ac_GROUP_COLORS)]
                     for i, g in enumerate(all_group_keys)}

        # ── Gather selected groups' display traces ──
        show_groups_data = OrderedDict()
        for g in all_group_keys:
            if g not in selected_keys:
                continue
            tids = groups[g]
            solo = g.startswith('_solo_')
            le = ov_legend_edits.get(tids[0])
            label_raw = le.value.strip() if le and le.value.strip() else (
                _ov_tname(tids[0]) if solo else f'[{g}]')
            disp_traces = []
            for tid in tids:
                disp_traces.extend(_emp_get_display_traces(tid, mode, do_trim, tmax))
            if disp_traces:
                show_groups_data[g] = {'label_raw': label_raw, 'traces': disp_traces}

        if not show_groups_data:
            print('❌ No traces found for selected groups.'); return

        # ── Plot ──
        fig, ax = plt.subplots(figsize=(11, 5.5), dpi=_ac_DPI)

        for g, d in show_groups_data.items():
            col = color_map[g]
            label_fmt = _ac_format_label_superscript(d['label_raw'])
            n_tr = len(d['traces'])

            for i, (x_disp, y_disp, name) in enumerate(d['traces']):
                lbl = f'{label_fmt} (n={n_tr})' if i == 0 else None
                ax.plot(x_disp, y_disp, color=col, lw=_ac_LW, alpha=0.75, label=lbl)

        # ── Threshold line and band ──
        ax.axhline(threshold, color='#ff4444', ls='--', lw=1.2, alpha=0.85, zorder=5)
        if threshold_sd > 0:
            ax.axhspan(threshold - threshold_sd, threshold + threshold_sd,
                       color='#ff4444', alpha=0.07, zorder=1)
            ax.axhline(threshold - threshold_sd, color='#ff4444', ls=':', lw=0.5, alpha=0.5, zorder=4)
            ax.axhline(threshold + threshold_sd, color='#ff4444', ls=':', lw=0.5, alpha=0.5, zorder=4)

        # Annotation inside plot area, below the threshold line, right side
        ax.annotate(f'Inflection Mean ± SD from {ref_label_fmt}\ny = {thresh_str}',
                    xy=(0.98, threshold), xycoords=('axes fraction', 'data'),
                    xytext=(0, -12), textcoords='offset points',
                    fontsize=7, color='#ff4444', fontweight='bold',
                    va='top', ha='right',
                    bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#ff4444', alpha=0.9),
                    zorder=7)

        ax.set_xlabel('Time (min)', fontsize=11)
        ax.set_ylabel('Norm.' if mode == 'Normalized (0-1)' else 'ΔmV', fontsize=11)

        if ac_title_w.value.strip():
            title = _ac_format_label_superscript(ac_title_w.value.strip())
        else:
            title = 'Individual Replicate Photodiode Voltage Traces over Time'

        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.grid(True, ls='--', alpha=0.3)
        ax.set_xlim(right=tmax)

        ax.legend(fontsize=8, loc='upper left', framealpha=0.9)

        plt.tight_layout()
        _ac_save_fig(fig, 'groups_threshold.png')
        plt.show(); plt.close(fig)

        # ── Tt crossing summary ──
        print(f'\nThreshold crossing times (y = {thresh_str}):')
        print(f'{"Group":<20} {"Mean Tt":>10} {"± SD":>10} {"n":>8}')
        print('─' * 52)
        for g, d in show_groups_data.items():
            label = d['label_raw']
            tts = []
            for x_disp, y_disp, name in d['traces']:
                tt = _emp_first_crossing(x_disp, y_disp, threshold)
                if tt is not None:
                    tts.append(tt)
            if tts:
                m = np.mean(tts)
                s = np.std(tts, ddof=1) if len(tts) > 1 else 0
                print(f'{label:<20} {m:>10.2f} {s:>10.2f} {len(tts):>5}/{len(d["traces"])}')
            else:
                print(f'{label:<20} {"DNF":>10} {"—":>10} {0:>5}/{len(d["traces"])}')

ac_btn.on_click(_ac_plot)
_ac_refresh()

display(widgets.VBox([
    widgets.HTML('<h3 style="color:#2E7D32;margin-bottom:4px">All Groups — Threshold Crossing View</h3>'
                 '<span style="color:#888;font-size:12px">Select which concentration groups to display. '
                 'Hold Ctrl/Cmd to pick multiple. Colors stay consistent regardless of selection. '
                 'Threshold is the inflection mean from the reference group.</span>'),
    widgets.HBox([ac_ref_dd, ac_refresh_btn]),
    widgets.HBox([ac_show_dd, widgets.VBox([
        ac_select_all_btn,
        widgets.HTML('<span style="color:#888;font-size:10px">Ctrl/Cmd+click<br>to multi-select</span>')
    ])]),
    ac_title_w,
    ac_btn,
    ac_out,
], layout=widgets.Layout(border='2px solid #2E7D32', padding='8px', margin='8px 0', border_radius='8px')))